## Foundry Agent Tracing with Application Insights

### Installing Libraries and SDKs

In [ ]:
%pip install azure-ai-projects==2.7.0 openai==1.109.1 python-dotenv azure-identity

### Setting up the Environment Variables

In [ ]:
import os
from azure.ai.projects.models import PromptAgentDefinition
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool
from opentelemetry.trace import (
    Status,
    StatusCode
)

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"

### Creating the Foundry Project Client

In [3]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Enable Tracing with Application Insights

In [ ]:
connection_string = project_client.telemetry.get_application_insights_connection_string()
configure_azure_monitor(connection_string=connection_string)

print("App Insights Connection String used: {}".format(connection_string))

App Insights Connection String: InstrumentationKey=2d086d55-142b-43ef-9ea7-d6ce8ddc7401;IngestionEndpoint=https://swedencentral-0.in.applicationinsights.azure.com/;LiveEndpoint=https://swedencentral.livediagnostics.monitor.azure.com/;ApplicationId=8295dad5-c2fd-44a6-bde7-5241f8733da5


### Creating the OpenAI Client

In [ ]:
openai_client = project_client.get_openai_client()

### Fetch the MCP Server Connection Reference

In [ ]:
connection_id = ""

for connection in project_client.connections.list():
    if connection.name == mcp_server_name:
        connection_id = connection.id
        break

print(f"The MCP Server Connection ID is: {connection_id}")

### Define the Complete End-to-End Trace

In [ ]:
tracer = trace.get_tracer(__name__)

with tracer.start_as_current_span(
    "microsoft-learn-agent-request"
) as root_span:

    root_span.set_attribute("app.name", "AI-500-Agent-Lab")

    root_span.set_attribute("scenario.name", "microsoft-learn-search")

    root_span.set_attribute("agent.name", "MCP-Tracing-Agent")

    root_span.set_attribute("agent.model", model_deployment_name)

    span_context = (root_span.get_span_context())

    trace_id = format(span_context.trace_id, "032x")

    with tracer.start_as_current_span( "prepare-agent") as span:

        span.set_attribute("tool.type", "mcp")

        span.set_attribute("mcp.server.name", "microsoft_learn_server")

        tool = MCPTool(
            server_label = "microsoft_learn_server",

            server_url = "https://learn.microsoft.com/api/mcp",

            require_approval = "never",

            project_connection_id = connection_id
        )


        span.add_event("MCP tool configured")

        agent = project_client.agents.create_version(
                agent_name = "MCP-Tracing-Agent",

                definition = PromptAgentDefinition(

                        model = model_deployment_name,

                        instructions="""
                        You are an intelligent assistant that can
                        interact with the Microsoft Learn MCP server
                        to provide users with relevant learning
                        resources and information about Microsoft
                        technologies.
                        """,

                        tools=[
                            tool
                        ],
                    )
        )

        span.set_attribute("agent.id", str(agent.id))

        span.add_event("Foundry agent created")

    with tracer.start_as_current_span("create-conversation") as span:

        conversation = openai_client.conversations.create()

        span.set_attribute("conversation.id", conversation.id)

        span.add_event("Conversation created")

        user_query = """
                    Can you help me with the latest
                    AI Foundry SDK code samples?
                    """


        with tracer.start_as_current_span("execute-microsoft-learn-agent") as span:

            span.set_attribute("agent.name", "MCP-Tracing-Agent")

            span.set_attribute("query.category", "foundry-sdk")

            span.set_attribute("conversation.id", conversation.id)

            span.add_event("Agent execution started")


            try:

                response = openai_client.responses.create(
                        conversation = conversation.id,
                        extra_body={
                            "agent_reference": {
                                "name": "MCP-Tracing-Agent",
                                "type": "agent_reference"
                            }
                        },
                        input = user_query
                    )

                span.set_attribute("response.success", True)

                span.set_status(Status(StatusCode.OK))

                span.add_event("Agent response received")


            except Exception as error:

                span.record_exception(error)

                span.set_attribute("response.success", False)

                span.set_status(Status(StatusCode.ERROR,str(error)))

                raise

        print("\nAgent Response:\n")

        print(response.output_text)